# Scientific validation of the deployment-4 rotation figure

These two analyses extend the final figure in `beam_modulation_d4.ipynb`. They use that notebook's raw-data/cache loading, selected tones, and selected turn, so no exploration-generated `tones.npy` or `turn.npy` is required. Run from within this repository with the final notebook's data available. The setup regenerates the final figure exports once; generated outputs remain ignored.

Recorded exploration results (not newly measured by this cleanup): raw depth tracked tone-to-continuum ratio with median absolute difference 0.9 dB and correlation 0.886. Continuum subtraction changed median depths below 160 MHz from 13.4 to 27.5 dB, and above 160 MHz from 22.7 to 29.1 dB. Low-band subtracted depths exceeded estimated continuum noise headroom, so they bound the null rather than accurately measure it.

The turn overlays placed the first null at -20.4 ± 0.0 s and the second minimum at +28.5 ± 2.7 s. Their unequal timing and shapes are consistent with nonuniform angular motion; without telemetry this is not a unique reconstruction of the kinematics. Broad second minima make their frequency-dependent argmin uncertain. The rotation curves provide no positive multipath diagnosis: a 590 ns terrain delay corresponds to roughly 1.7 MHz fringes, undersampled by the 3.906 MHz comb. Use the delay-spectrum analysis for that question. The final figure retains a time axis and raw-power reduction; these checks do not silently replace it with subtracted depths.


In [ ]:
from pathlib import Path
import json
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "notebooks/christian").is_dir():
    if REPO == REPO.parent:
        raise FileNotFoundError("Run from within the data-analysis repository")
    REPO = REPO.parent
OUT = REPO / "notebooks/christian/beam_modulation_d4"
__file__ = str(OUT / "validation.py")  # legacy diagnostics resolve outputs beside this notebook
_figure = None

def load_figure():
    """Run the final notebook's code in its own namespace, once per validation session."""
    global _figure
    if _figure is None:
        notebook = json.loads((OUT / "beam_modulation_d4.ipynb").read_text())
        namespace = {"__name__": "__validation__"}
        previous = Path.cwd()
        try:
            os.chdir(OUT)
            for index, cell in enumerate(notebook["cells"]):
                if cell["cell_type"] == "code":
                    exec(compile("".join(cell["source"]), f"final-notebook-cell-{index}", "exec"), namespace)
            _figure = namespace
        finally:
            os.chdir(previous)
            plt.close("all")
    return _figure


In [ ]:
g = load_figure()
D = dict(fr=g["freq"], ch=g["chan"], tsec=g["tsec"], sky=g["sky"], d=g["d"])
ROT, COMB_RESIDUE, COMB_SPILL = g["ROT"], g["COMB_RESIDUE"], g["COMB_SPILL"]
load = lambda: D
db = g["to_db"]
flanking = lambda c, nchan: g["flanking"](c)


## Continuum subtraction and noise headroom

Preserved from `explore/11_continuum.py`; diagnostic calculations retained.

In [ ]:
"""Are the shallow low-frequency shapes real, or just the additive continuum capping them?

Raw channel power is tone + sky + receiver noise.  When the tone is nulled the channel
only falls to the continuum, so the deepest drop a channel can show is capped by its
tone-to-continuum ratio at the peak.  Subtract the continuum (in linear power, as the
paper's 2026 figure does) and the cap goes away.
"""
import numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

D = load()
fr, ch, t = D["fr"], D["ch"], D["tsec"]
nchan = fr.size
tones = g["tones"]
A, B, E = (g["A"], g["B"], g["E"])
HALF = 0.25 * (B - A) + 0.25 * (E - B)
sel = D["sky"] & (np.abs(t - B) <= HALF)
tau = (t - B)[sel]
d = D["d"][ROT]

raw, sub, snr, noise = {}, {}, {}, {}
for c in tones:
    nb = flanking(c, nchan)
    tot = d[sel, c]
    cont = np.nanmedian(d[sel][:, nb], 1)
    ref = np.abs(tau) < 2.0
    raw[c] = db(tot) - db(np.nanmedian(tot[ref]))
    ex = tot - cont                                  # linear-power continuum subtraction
    sub[c] = db(np.where(ex > 0, ex, np.nan)) - db(np.nanmedian(ex[ref]))
    snr[c] = db(np.nanmedian(tot[ref])) - db(np.nanmedian(cont[ref]))
    # noise on the continuum estimate sets how deep a subtracted tone can be believed
    noise[c] = db(np.nanmedian(ex[ref])) - db(np.nanstd(cont))

dep = lambda p: np.nanmax(p) - np.nanmin(p)
print(f"{'MHz':>7} {'tone-cont':>10} {'raw depth':>10} {'sub depth':>10} {'headroom':>9}")
for c in tones:
    print(f"{fr[c]:7.1f} {snr[c]:10.1f} {dep(raw[c]):10.1f} {dep(sub[c]):10.1f} {noise[c]:9.1f}")

rawd = np.array([dep(raw[c]) for c in tones])
subd = np.array([dep(sub[c]) for c in tones])
s = np.array([snr[c] for c in tones])
print(f"\nraw depth vs tone-to-continuum: median |difference| {np.median(np.abs(rawd - s)):.1f} dB, "
      f"correlation {np.corrcoef(rawd, s)[0,1]:.3f}")
lo, hi = fr[tones] < 160, fr[tones] >= 160
print(f"below 160 MHz: raw {np.median(rawd[lo]):5.1f} dB -> subtracted {np.median(subd[lo]):5.1f} dB "
      f"(tone sits {np.median(s[lo]):.1f} dB over continuum)")
print(f"above 160 MHz: raw {np.median(rawd[hi]):5.1f} dB -> subtracted {np.median(subd[hi]):5.1f} dB "
      f"(tone sits {np.median(s[hi]):.1f} dB over continuum)")

norm = Normalize(fr[tones].min(), fr[tones].max())
smap = ScalarMappable(norm, plt.cm.plasma)
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
for c in tones:
    axs[0].plot(tau, raw[c], color=smap.to_rgba(fr[c]), lw=0.8)
    axs[1].plot(tau, sub[c], color=smap.to_rgba(fr[c]), lw=0.8)
axs[0].set_title("raw channel power", fontsize=10)
axs[1].set_title("continuum subtracted (as the 2026 paper figure)", fontsize=10)
for ax in axs[:2]:
    ax.set_xlim(-HALF, HALF); ax.grid(alpha=0.3)
    ax.set_xlabel("time through turn [s]"); ax.set_ylabel("dB rel. $t=0$")
axs[1].set_ylim(-45, 15)
axs[2].plot(s, rawd, "o", ms=4, label="raw")
axs[2].plot(s, subd, "s", ms=4, label="subtracted")
axs[2].plot([0, 28], [0, 28], "k:", lw=0.8, label="depth = tone-to-continuum")
axs[2].set_xlabel("tone above continuum at peak [dB]"); axs[2].set_ylabel("measured depth [dB]")
axs[2].legend(fontsize=8); axs[2].grid(alpha=0.3)
fig.colorbar(smap, ax=axs[1], label="MHz", pad=0.02)
fig.tight_layout(); fig.savefig(OUT / "11_continuum.png", dpi=110)


## Repeatability and asymmetric minima

Preserved from `explore/12_asymmetry.py`; diagnostic calculations retained.

In [ ]:
"""Is the second-half structure orientation-locked (beam/multipath) or kinematic?

Overlay every turn in the episode.  Structure fixed in the antenna frame repeats turn to
turn; structure that moves is the platform's rate varying.
"""
import numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

D = load()
fr, ch, t = D["fr"], D["ch"], D["tsec"]
nchan = fr.size
tones = g["tones"]
band = (fr > 150) & (fr < 195)
c = np.where(D["sky"],
             db(np.nanmedian(D["d"][ROT][:, (ch % 16 == COMB_RESIDUE) & band], 1))
             - db(np.nanmedian(D["d"][ROT][:, ~np.isin(ch % 16, COMB_SPILL) & band], 1)), np.nan)

k = (t > 640) & (t < 1530) & np.isfinite(c)
x, ys = t[k], uniform_filter1d(c[k], 5)
pk = [i for i in range(3, len(ys) - 3) if ys[i] == max(ys[i - 3:i + 4]) and ys[i] > 18.0]
tall = []
for i in pk:
    if not tall or x[i] - x[tall[-1]] > 20:
        tall.append(i)
    elif ys[i] > ys[tall[-1]]:
        tall[-1] = i
tm = x[tall]

print(f"{'centre':>8} {'halves [s]':>13} {'1st null':>9} {'2nd min':>8} {'depth1':>7} {'depth2':>7}")
rows = []
for n in range(1, len(tm) - 1):
    a, b, e = tm[n - 1], tm[n], tm[n + 1]
    if not (60 < b - a < 95 and 60 < e - b < 95):
        continue
    m = D["sky"] & (np.abs(t - b) <= 0.25 * (b - a) + 0.25 * (e - b)) & np.isfinite(c)
    tau, v = t[m] - b, uniform_filter1d(c[m], 3)
    neg, pos = tau < -4, tau > 4
    t1, t2 = tau[neg][np.argmin(v[neg])], tau[pos][np.argmin(v[pos])]
    d1, d2 = v[np.abs(tau) < 2].max() - v[neg].min(), v[np.abs(tau) < 2].max() - v[pos].min()
    rows.append((b, tau, v, t1, t2))
    print(f"{b:8.1f} {b-a:6.1f}/{e-b:5.1f} {t1:9.1f} {t2:8.1f} {d1:7.1f} {d2:7.1f}")

t1s = np.array([r[3] for r in rows]); t2s = np.array([r[4] for r in rows])
print(f"\n1st null at {t1s.mean():.1f} +- {t1s.std():.1f} s   "
      f"2nd minimum at {t2s.mean():+.1f} +- {t2s.std():.1f} s")
print(f"|t1| vs t2: {np.abs(t1s).mean():.1f} vs {t2s.mean():.1f} s "
      f"-> the two minima are NOT symmetric about the maximum")

# per-tone: where is the second minimum, as a function of frequency?
A, B, E = (g["A"], g["B"], g["E"])
HALF = 0.25 * (B - A) + 0.25 * (E - B)
sel = D["sky"] & (np.abs(t - B) <= HALF)
tau = (t - B)[sel]
print(f"\n{'MHz':>7} {'1st null t':>11} {'2nd min t':>10}  (continuum-subtracted)")
for cc in tones[::4]:
    tot = D["d"][ROT][sel, cc]
    ex = tot - np.nanmedian(D["d"][ROT][sel][:, flanking(cc, nchan)], 1)
    v = uniform_filter1d(db(np.where(ex > 0, ex, np.nan)), 3)
    neg, pos = tau < -4, tau > 4
    print(f"{fr[cc]:7.1f} {tau[neg][np.nanargmin(v[neg])]:11.1f} {tau[pos][np.nanargmin(v[pos])]:10.1f}")

fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
for b, tau_, v, *_ in rows:
    axs[0].plot(tau_, v, lw=0.9, label=f"t={b:.0f} s")
axs[0].set_xlabel("time from maximum [s]"); axs[0].set_ylabel("comb contrast [dB]")
axs[0].legend(fontsize=7); axs[0].grid(alpha=0.3); axs[0].set_title("every usable turn overlaid", fontsize=10)
allt = np.concatenate([r[1] for r in rows]); allv = np.concatenate([r[2] for r in rows])
axs[1].plot(allt, allv, ".", ms=3, alpha=0.6)
axs[1].set_xlabel("time from maximum [s]"); axs[1].set_ylabel("comb contrast [dB]")
axs[1].grid(alpha=0.3); axs[1].set_title("pooled", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "12_asymmetry.png", dpi=110)
